### Data Quality & Validation Framework

**What this notebook does:**
```
PostgreSQL  →  blood_reports
PostgreSQL  →  dicom              ─┐
                                   ├─► Great Expectations checks
TimescaleDB →  wearable_readings  ─┘        │
                                            ▼
                              data_quality_log   (all results)
                              data_quarantine    (rejected records)
```

**Quality checks performed:**
```
✔ Missing patient IDs
✔ Invalid / missing dates
✔ Duplicate records
✔ Out-of-range values (glucose, heart rate etc.)
✔ Null / corrupted rows
```

**Re-run safe:** Every run gets a unique `run_id` — previous results are never overwritten.

**Tables needed:**
```
data_quality_log   → one row per check per run  (PostgreSQL)
data_quarantine    → one row per rejected record (PostgreSQL)
```

## 1. Install & Import

In [ ]:
#!pip install great-expectations psycopg2-binary pandas python-dotenv --quiet

In [1]:
import os
import json
import psycopg2
import pandas as pd
import great_expectations as gx
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
import json   

In [3]:

def get_pg():
    """PostgreSQL — blood_reports + quality tables."""
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

def get_ts():
    """TimescaleDB — wearable_readings."""
    return psycopg2.connect(
        host="localhost", port=5433,
        dbname=os.getenv("TIMESCALE_DB"),
        user=os.getenv("TIMESCALE_USER"),
        password=os.getenv("TIMESCALE_PASSWORD")
    )

pg = get_pg()
ts = get_ts()
print("✅ Connected")

✅ Connected


---
## 2. Create Quality Tables — Run Once

In [4]:
def create_quality_tables(pg):
    with pg.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS data_quality_log (
                id           SERIAL PRIMARY KEY,
                run_id       TEXT        NOT NULL,
                run_at       TIMESTAMPTZ DEFAULT NOW(),
                source       TEXT,           -- blood_reports | wearable_readings
                check_name   TEXT,           -- name of the check
                status       TEXT,           -- PASS | FAIL
                rows_checked INT,
                rows_failed  INT,
                details      TEXT
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS data_quarantine (
                id             SERIAL PRIMARY KEY,
                run_id         TEXT        NOT NULL,
                quarantined_at TIMESTAMPTZ DEFAULT NOW(),
                source         TEXT,
                check_name     TEXT,
                record_id      TEXT,
                reason         TEXT,
                raw_data       JSONB
            );
        """)

    pg.commit()
    print("✅ data_quality_log and data_quarantine tables ready")


create_quality_tables(pg)

✅ data_quality_log and data_quarantine tables ready


---
## 3. Helpers — Log Result & Quarantine Bad Rows

In [5]:
def log_result(pg, run_id, source, check_name, status, rows_checked, rows_failed, details=""):
    """Write one check result to data_quality_log."""
    with pg.cursor() as cur:
        cur.execute("""
            INSERT INTO data_quality_log
                (run_id, source, check_name, status, rows_checked, rows_failed, details)
            VALUES (%s,%s,%s,%s,%s,%s,%s)
        """, (run_id, source, check_name, status, rows_checked, rows_failed, details))
    pg.commit()
    icon = "✅" if status == "PASS" else "❌"
    print(f"    {icon} {check_name:<45} {status}  ({rows_failed}/{rows_checked} failed)")


def quarantine_rows(pg, run_id, source, check_name, bad_df):
    """Store rejected rows into data_quarantine."""
    with pg.cursor() as cur:
        for _, row in bad_df.iterrows():
            raw = json.loads(row.to_json())
            cur.execute("""
                INSERT INTO data_quarantine
                    (run_id, source, check_name, record_id, reason, raw_data)
                VALUES (%s,%s,%s,%s,%s,%s)
            """, (
                run_id, source, check_name,
                str(row.get("customer_id", "unknown")),
                check_name,
                # json.dumps(row.astype(str).to_dict())
                json.dumps(raw)
            ))
    pg.commit()
    print(f"       ⚠️  {len(bad_df)} record(s) quarantined")

---
## 4. Define Value Ranges
Medically plausible ranges for blood markers and wearable sensors.

In [6]:
# (min, max) — values outside these ranges are flagged
BLOOD_RANGES = {
    "glucose":           (20,   600),
    "hemoglobin":        (3,    25),
    "cholesterol_total": (50,   500),
    "cholesterol_hdl":   (10,   150),
    "cholesterol_ldl":   (10,   400),
    "triglycerides":     (20,   2000),
    "wbc":               (0.5,  100),
    "rbc":               (1,    10),
}

WEARABLE_RANGES = {
    "heart_rate":        (30,   220),
    "spo2_pct":          (70,   100),
    "steps":             (0,    100000),
    "skin_temp_c":       (30,   42),
    "hrv_ms":            (1,    300),
    "respiratory_rate":  (5,    60),
}

print("✅ Ranges defined")

✅ Ranges defined


---
## 5. Great Expectations — Blood Reports

Checks: missing patient ID, missing date, duplicate minio_path, out-of-range markers.

In [7]:
def check_blood_reports(pg, run_id):
    SOURCE = "blood_reports"
    df     = pd.read_sql("SELECT * FROM blood_reports", pg)
    print(f"\n── Blood Reports ({len(df)} rows) ────────────────")

    # build GX context
    ctx   = gx.get_context(mode="ephemeral")
    ds    = ctx.data_sources.add_pandas("blood_ds")
    da    = ds.add_dataframe_asset("blood_asset")
    # batch = da.add_batch_definition_whole_dataframe("blood_batch").get_batch(
    #             batch_parameters={"dataframe": df})
    batch_def = da.add_batch_definition_whole_dataframe("blood_batch")
    # batch = 

    
    # batch = batch_def.get_batch(batch_parameters={"dataframe": df}) 

    
    
    suite = ctx.suites.add(gx.ExpectationSuite(name="blood_suite"))
    

    # ── expectations ─────────────────────────────────────────────────
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="report_date"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="minio_path"))
    for col, (lo, hi) in BLOOD_RANGES.items():
        if col in df.columns:
            suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
                column=col, min_value=lo, max_value=hi, mostly=1
            ))

    # ── run ──────────────────────────────────────────────────────────
    # vd  = ctx.validation_definitions.add(
    #         gx.ValidationDefinition(name="blood_vd", data=batch, suite=suite))
    vd  = ctx.validation_definitions.add(
        gx.ValidationDefinition(name="blood_vd", data=batch_def, suite=suite))  

    res = vd.run(batch_parameters={"dataframe": df})  # ✅


    # # res = vd.run()
    # res = vd.run(batch_parameters={"dataframe": df})

    # ── log each check ───────────────────────────────────────────────
    for r in res.results:
        col      = r.expectation_config.kwargs.get("column", "")
        name     = f"{r.expectation_config.type}:{col}" if col else r.expectation_config.type
        status   = "PASS" if r.success else "FAIL"
        checked  = r.result.get("element_count", len(df))
        failed   = r.result.get("unexpected_count", 0)

        log_result(pg, run_id, SOURCE, name, status, checked, failed)

        # quarantine bad rows on failure
        if not r.success and col in df.columns:
            bad = df[df[col].isna()]
            if not bad.empty:
                quarantine_rows(pg, run_id, SOURCE, name, bad)

    passed = res.statistics["successful_expectations"]
    total  = res.statistics["evaluated_expectations"]
    print(f"  📋 {passed}/{total} checks passed")


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"🚀 Run ID: {RUN_ID}")
check_blood_reports(pg, RUN_ID)

🚀 Run ID: 20260516_050913

── Blood Reports (15 rows) ────────────────


Calculating Metrics: 100%|██████████| 76/76 [00:00<00:00, 1920.63it/s]

    ✅ expect_column_values_to_not_be_null:customer_id PASS  (0/15 failed)
    ❌ expect_column_values_to_not_be_null:report_date FAIL  (5/15 failed)
       ⚠️  5 record(s) quarantined
    ❌ expect_column_values_to_be_unique:minio_path  FAIL  (15/15 failed)
    ✅ expect_column_values_to_be_between:glucose    PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:hemoglobin PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_total PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_hdl PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_ldl PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:triglycerides PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:wbc        PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:rbc        PASS  (0/15 failed)
  📋 9/11 checks passed


---
## 6. Great Expectations — Wearable Readings

In [12]:
def check_wearable_readings(ts, pg, run_id):
    SOURCE = "wearable_readings"
    df     = pd.read_sql("SELECT * FROM wearable_readings", ts)
    print(f"\n── Wearable Readings ({len(df)} rows) ───────────────")

    # build GX context
    ctx   = gx.get_context(mode="ephemeral")
    ds    = ctx.data_sources.add_pandas("wearable_ds")
    da    = ds.add_dataframe_asset("wearable_asset")
    batch_def = da.add_batch_definition_whole_dataframe("wearable_batch")  # ✅ batch_def not batch

    suite = ctx.suites.add(gx.ExpectationSuite(name="wearable_suite"))

    # ── expectations ─────────────────────────────────────────────────
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="time"))
    suite.add_expectation(gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["time", "customer_id"]))
    for col, (lo, hi) in WEARABLE_RANGES.items():
        if col in df.columns:
            suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
                column=col, min_value=lo, max_value=hi, mostly=1
            ))

    # ── run ──────────────────────────────────────────────────────────
    vd  = ctx.validation_definitions.add(
            gx.ValidationDefinition(name="wearable_vd", data=batch_def, suite=suite))  # ✅ batch_def
    res = vd.run(batch_parameters={"dataframe": df})  # ✅ df passed here

    # ── log each check ───────────────────────────────────────────────
    for r in res.results:
        col     = r.expectation_config.kwargs.get("column", "")
        name    = f"{r.expectation_config.type}:{col}" if col else r.expectation_config.type
        status  = "PASS" if r.success else "FAIL"
        checked = r.result.get("element_count", len(df))
        failed  = r.result.get("unexpected_count", 0)

        log_result(pg, run_id, SOURCE, name, status, checked, failed)

        if not r.success and col in df.columns:
            bad = df[df[col].isna()]
            if not bad.empty:
                quarantine_rows(pg, run_id, SOURCE, name, bad)

    passed = res.statistics["successful_expectations"]
    total  = res.statistics["evaluated_expectations"]
    print(f"  📋 {passed}/{total} checks passed")


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"🚀 Run ID: {RUN_ID}")
check_wearable_readings(ts, pg, RUN_ID)

🚀 Run ID: 20260516_055511

── Wearable Readings (3600 rows) ───────────────


Calculating Metrics: 100%|██████████| 61/61 [00:00<00:00, 997.03it/s] 

    ✅ expect_column_values_to_not_be_null:customer_id PASS  (0/3600 failed)
    ✅ expect_column_values_to_not_be_null:time      PASS  (0/3600 failed)
    ✅ expect_compound_columns_to_be_unique          PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:heart_rate PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:spo2_pct   PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:steps      PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:skin_temp_c PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:hrv_ms     PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:respiratory_rate PASS  (0/3600 failed)
  📋 9/9 checks passed


---
## 7. ▶️ Run All Checks — Use This Every Time New Data Is Added

This reloads fresh data from both databases and runs all checks.
Previous run results are kept — nothing is overwritten.

In [13]:
def run_all_checks():
    pg_conn = get_pg()
    ts_conn = get_ts()
    run_id  = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    print(f"🚀 Quality check run: {run_id}")
    check_blood_reports(pg_conn, run_id)
    check_wearable_readings(ts_conn, pg_conn, run_id)

    pg_conn.close()
    ts_conn.close()
    print("\n✅ All checks complete")
    return run_id


RUN_ID = run_all_checks()

🚀 Quality check run: 20260516_055525

── Blood Reports (15 rows) ────────────────


Calculating Metrics: 100%|██████████| 76/76 [00:00<00:00, 2575.54it/s]

    ✅ expect_column_values_to_not_be_null:customer_id PASS  (0/15 failed)
    ❌ expect_column_values_to_not_be_null:report_date FAIL  (5/15 failed)
       ⚠️  5 record(s) quarantined
    ❌ expect_column_values_to_be_unique:minio_path  FAIL  (15/15 failed)
    ✅ expect_column_values_to_be_between:glucose    PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:hemoglobin PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_total PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_hdl PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:cholesterol_ldl PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:triglycerides PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:wbc        PASS  (0/15 failed)
    ✅ expect_column_values_to_be_between:rbc        PASS  (0/15 failed)
  📋 9/11 checks passed

── Wearable Readings (3600 rows) ───────────────



Calculating Metrics: 100%|██████████| 61/61 [00:00<00:00, 2074.20it/s]


    ✅ expect_column_values_to_not_be_null:customer_id PASS  (0/3600 failed)
    ✅ expect_column_values_to_not_be_null:time      PASS  (0/3600 failed)
    ✅ expect_compound_columns_to_be_unique          PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:heart_rate PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:spo2_pct   PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:steps      PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:skin_temp_c PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:hrv_ms     PASS  (0/3600 failed)
    ✅ expect_column_values_to_be_between:respiratory_rate PASS  (0/3600 failed)
  📋 9/9 checks passed

✅ All checks complete


In [20]:
#### Visual HTML report

In [27]:
def build_data_docs(df, suite_name, vd_name, source="blood"):
    """
    Run GX validation and open the visual HTML Data Docs report.
    source: "blood" or "wearable"
    """
    ctx   = gx.get_context(mode="ephemeral")
    ds    = ctx.data_sources.add_pandas("docs_ds")
    da    = ds.add_dataframe_asset("docs_asset")
    batch_def = da.add_batch_definition_whole_dataframe("docs_batch")  # ✅

    suite = ctx.suites.add(gx.ExpectationSuite(name=suite_name))

    if source == "blood":
        # ── blood report expectations ─────────────────────────────────
        suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
        suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="report_date"))
        suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="minio_path"))
        for col, (lo, hi) in BLOOD_RANGES.items():
            if col in df.columns:
                suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
                    column=col, min_value=lo, max_value=hi, mostly=1
                ))

    elif source == "wearable":
        # ── wearable expectations ─────────────────────────────────────
        suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
        suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="time"))
        suite.add_expectation(gx.expectations.ExpectCompoundColumnsToBeUnique(
            column_list=["time", "customer_id"]))
        for col, (lo, hi) in WEARABLE_RANGES.items():
            if col in df.columns:
                suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
                    column=col, min_value=lo, max_value=hi, mostly=1
                ))

    vd = ctx.validation_definitions.add(
            gx.ValidationDefinition(name=vd_name, data=batch_def, suite=suite))  # ✅

    # ── checkpoint generates the visual report ────────────────────────
    checkpoint = ctx.checkpoints.add(gx.Checkpoint(
        name                   = f"{vd_name}_checkpoint",
        validation_definitions = [vd],
        actions                = [gx.checkpoint.UpdateDataDocsAction(name="update_docs")]
    ))

    result = checkpoint.run(batch_parameters={"dataframe": df})  # ✅

    ctx.open_data_docs()
    print(f"✅ Data Docs opened in browser for: {source}")
    return result


# ── run for blood reports ─────────────────────────────────────────────
df_blood = pd.read_sql("SELECT * FROM blood_reports", pg)
build_data_docs(df_blood, "blood_suite_docs", "blood_vd_docs", source="blood")

# ── run for wearable readings ─────────────────────────────────────────
df_wearable = pd.read_sql("SELECT * FROM wearable_readings", ts)
build_data_docs(df_wearable, "wearable_suite_docs", "wearable_vd_docs", source="wearable")

Calculating Metrics: 100%|██████████| 76/76 [00:00<00:00, 3168.60it/s]


✅ Data Docs opened in browser for: blood


Calculating Metrics: 100%|██████████| 61/61 [00:00<00:00, 864.56it/s] 


✅ Data Docs opened in browser for: wearable


CheckpointResult(run_id={"run_name": null, "run_time": "2026-05-16T12:15:52.751093+05:30"}, run_results={ValidationResultIdentifier::wearable_suite_docs/__none__/20260516T064552.751093Z/docs_ds-docs_asset: {
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "docs_ds-docs_asset",
          "column": "customer_id"
        },
        "meta": {},
        "id": "1a78720e-31fb-43bc-87a5-6137c4e7a6f3",
        "severity": "critical"
      },
      "result": {
        "element_count": 3600,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
  

---
## 8. Quality Dashboard — Pass/Fail Summary

In [15]:
def quality_dashboard(pg):

    # pass/fail rates by source
    summary = pd.read_sql("""
        SELECT
            source,
            COUNT(*)                                                      AS total_checks,
            SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END)               AS passed,
            SUM(CASE WHEN status='FAIL' THEN 1 ELSE 0 END)               AS failed,
            ROUND(100.0 *
                SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END) / COUNT(*)
            , 1)                                                          AS pass_rate_pct
        FROM data_quality_log
        WHERE run_id = (SELECT MAX(run_id) FROM data_quality_log)
        GROUP BY source
    """, pg)

    # failed checks
    failures = pd.read_sql("""
        SELECT source, check_name, rows_checked, rows_failed
        FROM data_quality_log
        WHERE status = 'FAIL'
          AND run_id = (SELECT MAX(run_id) FROM data_quality_log)
        ORDER BY source, rows_failed DESC
    """, pg)

    # quarantine summary
    quarantine = pd.read_sql("""
        SELECT source, check_name, COUNT(*) AS quarantined_rows
        FROM data_quarantine
        WHERE run_id = (SELECT MAX(run_id) FROM data_quality_log)
        GROUP BY source, check_name
        ORDER BY quarantined_rows DESC
    """, pg)

    print("\n" + "═"*55)
    print("  📊 QUALITY DASHBOARD — Latest Run")
    print("═"*55)

    print("\n  Pass/Fail by Source:")
    print(summary.to_string(index=False))

    print("\n  Failed Checks:")
    print(failures.to_string(index=False) if not failures.empty else "  None ✅")

    print("\n  Quarantined Records:")
    print(quarantine.to_string(index=False) if not quarantine.empty else "  None ✅")


quality_dashboard(pg)


═══════════════════════════════════════════════════════
  📊 QUALITY DASHBOARD — Latest Run
═══════════════════════════════════════════════════════

  Pass/Fail by Source:
           source  total_checks  passed  failed  pass_rate_pct
    blood_reports            11       9       2           81.8
wearable_readings             9       9       0          100.0

  Failed Checks:
       source                                      check_name  rows_checked  rows_failed
blood_reports    expect_column_values_to_be_unique:minio_path            15           15
blood_reports expect_column_values_to_not_be_null:report_date            15            5

  Quarantined Records:
       source                                      check_name  quarantined_rows
blood_reports expect_column_values_to_not_be_null:report_date                 5


---
## 9. Inspect Quarantined Records

In [16]:
def show_quarantine(pg):
    df = pd.read_sql("""
        SELECT quarantined_at, source, check_name, record_id, reason
        FROM data_quarantine
        ORDER BY quarantined_at DESC
        LIMIT 20
    """, pg)
    print("🚫 Quarantined Records (latest 20):")
    print(df.to_string(index=False))


show_quarantine(pg)

🚫 Quarantined Records (latest 20):
                  quarantined_at        source                                      check_name                    record_id                                          reason
2026-05-16 05:55:26.041496+00:00 blood_reports expect_column_values_to_not_be_null:report_date        CUST_mei_lin_66CBFE0F expect_column_values_to_not_be_null:report_date
2026-05-16 05:55:26.041496+00:00 blood_reports expect_column_values_to_not_be_null:report_date    CUST_amara_patel_03630F04 expect_column_values_to_not_be_null:report_date
2026-05-16 05:55:26.041496+00:00 blood_reports expect_column_values_to_not_be_null:report_date  CUST_carlos_rivera_5A81755B expect_column_values_to_not_be_null:report_date
2026-05-16 05:55:26.041496+00:00 blood_reports expect_column_values_to_not_be_null:report_date CUST_fatima_alsayed_8594AA83 expect_column_values_to_not_be_null:report_date
2026-05-16 05:55:26.041496+00:00 blood_reports expect_column_values_to_not_be_null:report_date CUST_john_

---
## 10. Full History — All Previous Runs

In [17]:
def show_history(pg):
    df = pd.read_sql("""
        SELECT
            run_id,
            source,
            SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END) AS passed,
            SUM(CASE WHEN status='FAIL' THEN 1 ELSE 0 END) AS failed,
            ROUND(100.0 *
                SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END) / COUNT(*)
            , 1)                                            AS pass_rate_pct
        FROM data_quality_log
        GROUP BY run_id, source
        ORDER BY run_id DESC
    """, pg)
    print("📅 Quality Check History — All Runs:")
    print(df.to_string(index=False))


show_history(pg)

📅 Quality Check History — All Runs:
         run_id            source  passed  failed  pass_rate_pct
20260516_055525 wearable_readings       9       0          100.0
20260516_055525     blood_reports       9       2           81.8
20260516_055511 wearable_readings       9       0          100.0
20260516_050913     blood_reports       9       2           81.8
20260516_050018     blood_reports       1       1           50.0
20260516_045544     blood_reports       1       1           50.0
20260516_043800     blood_reports       1       1           50.0
